In [24]:
import pandas as pd
import re
import random
import jieba
import nltk
nltk.download('punkt')  # Required for fallback if needed
from nltk.tokenize import sent_tokenize

[nltk_data] Downloading package punkt to /Users/carmenk/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [25]:

# === 1. 读取原始数据 ===
df = pd.read_excel("data/uni_infodisclosure_reports.xlsx")

In [26]:

# === 2. Clean appendix and footer ===
def clean_text(text):
    if not isinstance(text, str):
        return ""

    # Remove everything after '附件' only if followed by numbered list or URL
    appendix_match = re.search(r"\n\s*附件.*", text)
    if appendix_match:
        post_text = text[appendix_match.start():]
        if re.search(r"(http|www|\d+[\.\、])", post_text):
            text = text[:appendix_match.start()]

    # Remove footers like '上一条', '版权所有', etc.
    footer_patterns = [
        r"上一条.*",
        r"下一条.*",
        r"版权所有.*",
        r"Copyright.*",
        r"电话[:：].*",
        r"EMAIL[:：].*",
        r"【关闭】.*",
        r"index\.htm.*"
    ]
    for p in footer_patterns:
        text = re.sub(p, "", text, flags=re.IGNORECASE)

    return text

df["cleaned_text"] = df["text"].apply(clean_text)

# === 3. Sentence-level splitting and snippet grouping with length check ===
def split_sentences(text):
    text = re.sub(r"\s+", "", str(text))
    sentences = re.split(r'(?<=[。！？])', text)
    return [s for s in sentences if s.strip()]

def group_sentences(sentences, min_sents=4, max_sents=6, min_chars=400):
    grouped = []
    i = 0
    while i < len(sentences):
        current_snippet = ''
        count = 0
        while i < len(sentences) and (count < min_sents or len(current_snippet) < min_chars):
            current_snippet += sentences[i]
            i += 1
            count += 1
            if count >= max_sents:
                break
        if current_snippet.strip():
            grouped.append(current_snippet)
    return grouped

# === 4. Build final snippets dataframe ===
rows = []

for idx, row in df.iterrows():
    if not row["cleaned_text"] or pd.isna(row["cleaned_text"]):
        continue  # Skip blank or NaN text

    base_id = str(row["doc_id"]).strip()
    sents = split_sentences(row["cleaned_text"])
    snippets = group_sentences(sents, min_sents=4, max_sents=6, min_chars=400)

    for i, snip in enumerate(snippets, 1):
        row_data = row.drop(labels=["text", "cleaned_text", "doc_id"]).to_dict()
        row_data["doc_id"] = f"{base_id}_{str(i).zfill(2)}"
        row_data["text"] = snip
        rows.append(row_data)

# === 5. Finalize dataframe ===
final_df = pd.DataFrame(rows)
cols = ["doc_id", "text"] + [col for col in final_df.columns if col not in ["doc_id", "text"]]
final_df = final_df[cols]

# === 6. Save as UTF-8-SIG ===
final_df.to_csv("snippets_longer_min400chars.csv", index=False, encoding="utf-8-sig")
